## Regression Models that Evaluate all 5 Optimizers on Every Circuit
## and Select Optimizer with Highest Predicted 2Q Gate Reduction

In [ ]:
import pandas as pd
master_df = pd.read_csv("./dbs/original_data/single_opt_master.csv")
master_df.head()

In [ ]:
# Adds a percentage two qubit gate reduction column to the master dataframe
master_df["two_qubit_gate_reduction_pct"] = (
    (master_df["initial_two_qubit_gates"] - master_df["final_two_qubit_gates"])
    / master_df["initial_two_qubit_gates"]
    * 100
)

In [ ]:
#Labels and features for the best optimization runs
y = master_df["two_qubit_gate_reduction_pct"]
X = master_df[
    [
    "opt_1",
    "original_category",
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
    ]
]

In [ ]:
#one hot encoding for the categorical features
X = pd.get_dummies(X, columns=["opt_1", "original_category"], drop_first=False, dtype=int)

In [ ]:
#Circuit names for mapping back to master df
circuit_names = master_df["original_circuit"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(X, y, circuit_names, test_size=0.2, random_state=42)

# Random Forest Training and Evaluation

In [ ]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestRegressor
rf_regressor = RandomForestRegressor(n_estimators=50, random_state=42)

In [ ]:
#Training rl classifier
rf_regressor.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)

y_pred = rf_regressor.predict(X_train)

print("Training Data Metrics:")
print("R²:", r2_score(y_train, y_pred))
print("MAE:", mean_absolute_error(y_train, y_pred))
print("RMSE:", root_mean_squared_error(y_train, y_pred))

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)

y_pred = rf_regressor.predict(X_test)

print("Test Data Metrics:")
print("R²:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report

preds = rf_regressor.predict(X_test)


In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_regressor.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

In [ ]:
# Evaluating average 2Q gate depth reduction using RF regressor
# by trying all possible optimizers and choosing the predicted best one

total_reduction = 0
chosen_optimizers = []
results = []

optimizers = [
    "qiskit_ai",
    "qiskit_standard",
    "tket",
    "wisq_bqskit",
    "wisq_rules"
]

feature_cols = [
    "opt_1",
    "original_category",
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
]

for circuit_name in names_test.unique():
    circuit_rows = master_df[master_df["original_circuit"] == circuit_name]

    predictions = []

    for opt in optimizers:
        opt_rows = circuit_rows[circuit_rows["opt_1"] == opt]

        if opt_rows.empty:
            continue

        candidate = opt_rows.iloc[0].copy()

        X_candidate = pd.DataFrame([candidate[feature_cols]])

        X_candidate = pd.get_dummies(
            X_candidate,
            columns=["opt_1", "original_category"],
            drop_first=False,
            dtype=int
        )

        X_candidate = X_candidate.reindex(columns=X_train.columns, fill_value=0)

        pred_reduction = rf_regressor.predict(X_candidate)[0]

        predictions.append((opt, pred_reduction))

    if len(predictions) == 0:
        continue

    # Choose optimizer with highest predicted reduction
    best_opt, best_pred_reduction = max(predictions, key=lambda x: x[1])
    chosen_optimizers.append(best_opt)

    # Look up actual performance of chosen optimizer
    actual_row = circuit_rows[circuit_rows["opt_1"] == best_opt].iloc[0]

    original_gates = actual_row["initial_two_qubit_gates"]
    final_gates = actual_row["final_two_qubit_gates"]

    actual_reduction_pct = (
        (original_gates - final_gates)
        / original_gates
        * 100
    )

    total_reduction += actual_reduction_pct

    results.append({
        "original_circuit": circuit_name,
        "chosen_optimizer": best_opt,
        "predicted_reduction_pct": best_pred_reduction,
        "actual_reduction_pct": actual_reduction_pct
    })

average_reduction_RF = total_reduction / len(results)

print("Average reduction from RF-selected optimizers:", average_reduction_RF)

rf_selection_results = pd.DataFrame(results)
rf_selection_results.head()

In [ ]:
rf_selection_results["chosen_optimizer"].value_counts()

# XGBoost

In [ ]:
from xgboost import XGBRegressor
xgb_regressor = XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42
)

In [ ]:
xgb_regressor.fit(X_train, y_train)

In [ ]:
#Training percentage
train_score = xgb_regressor.score(X_train, y_train)
train_score

In [ ]:
#Testing percentage
test_score = xgb_regressor.score(X_test, y_test)
test_score

In [ ]:

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_regressor.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

In [ ]:
# Evaluating average 2Q gate depth reduction using RF regressor
# by trying all possible optimizers and choosing the predicted best one

total_reduction = 0
chosen_optimizers = []
results = []

optimizers = [
    "qiskit_ai",
    "qiskit_standard",
    "tket",
    "wisq_bqskit",
    "wisq_rules"
]

feature_cols = [
    "opt_1",
    "original_category",
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
]

for circuit_name in names_test.unique():
    circuit_rows = master_df[master_df["original_circuit"] == circuit_name]

    predictions = []

    for opt in optimizers:
        opt_rows = circuit_rows[circuit_rows["opt_1"] == opt]

        if opt_rows.empty:
            continue

        candidate = opt_rows.iloc[0].copy()

        X_candidate = pd.DataFrame([candidate[feature_cols]])

        X_candidate = pd.get_dummies(
            X_candidate,
            columns=["opt_1", "original_category"],
            drop_first=False,
            dtype=int
        )

        X_candidate = X_candidate.reindex(columns=X_train.columns, fill_value=0)

        pred_reduction = xgb_regressor.predict(X_candidate)[0]

        predictions.append((opt, pred_reduction))

    if len(predictions) == 0:
        continue

    # Choose optimizer with highest predicted reduction
    best_opt, best_pred_reduction = max(predictions, key=lambda x: x[1])
    chosen_optimizers.append(best_opt)

    # Look up actual performance of chosen optimizer
    actual_row = circuit_rows[circuit_rows["opt_1"] == best_opt].iloc[0]

    original_gates = actual_row["initial_two_qubit_gates"]
    final_gates = actual_row["final_two_qubit_gates"]

    actual_reduction_pct = (
        (original_gates - final_gates)
        / original_gates
        * 100
    )

    total_reduction += actual_reduction_pct

    results.append({
        "original_circuit": circuit_name,
        "chosen_optimizer": best_opt,
        "predicted_reduction_pct": best_pred_reduction,
        "actual_reduction_pct": actual_reduction_pct
    })

average_reduction_xgb = total_reduction / len(results)

print("Average reduction from XGB-selected optimizers:", average_reduction_xgb)

xgb_selection_results = pd.DataFrame(results)
xgb_selection_results.head()

In [ ]:
xgb_selection_results["chosen_optimizer"].value_counts()